In [43]:
import os
from dotenv import load_dotenv

from typing import Annotated, List
from langchain_core.tools import tool, StructuredTool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper
from langchain_core.messages import HumanMessage

from langchain.chat_models import init_chat_model

load_dotenv()

True

### **Building tools**

In [14]:
@tool
def division(a:int, b:int) -> int:
    """Divide 2 numbers"""
    return a / b

@tool
async def amultiply(a:int, b:int) -> int:
    "Multiply 2 numbers"
    return a * b

@tool
def multiply_by_max(
    a: Annotated[int, "A value"],
    b: Annotated[List[int], "List of ints over which the maximum value will be used"]
) -> int:
    """Multiply a with the maximum b"""
    return a * max(b)

In [11]:
print(division.name)
print(division.description)
print(division.args)

division
Divide 2 numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [12]:
print(multiply_by_max.args)

{'a': {'description': 'A value', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'List of ints over which the maximum value will be used', 'items': {'type': 'integer'}, 'title': 'B', 'type': 'array'}}


#### **Structured tool**

In [ ]:
@tool
def multiply(a:int, b:int) -> int:
    """Multiply 2 numbers"""
    return a * b

async def amultiply(a:int, b:int) -> int:
    "Multiply 2 numbers"
    return a * b

calculator = StructuredTool.from_function(func=multiply, coroutine=amultiply)

print(calculator.invoke({"a": 2, "b": 3}))
print(await calculator.ainvoke({"a": 2, "b": 5}))

#### **In-built tools**

In [16]:
wiki_api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=100)
wiki_tool = WikipediaQueryRun(api_wrapper=wiki_api_wrapper)

print(wiki_tool.run({
    "query": "langchain"
}))

Page: LangChain
Summary: LangChain is a software framework that helps facilitate the integration of 


In [ ]:
arxiv_tool = ArxivAPIWrapper()

arxiv_tool.run({"query": "1706.03762"})

### **Tool calling through LLMs**

In [26]:
tools = [wiki_tool, division, multiply]

In [27]:
llm = init_chat_model(model="groq:qwen/qwen3-32b").bind_tools(tools)
llm

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001E39FCF03E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E39FCF0D60>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'wikipedia', 'description': 'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.', 'parameters': {'properties': {'query': {'description': 'query to look up on wikipedia', 'type': 'string'}}, 'r

In [45]:
messages = [HumanMessage("What is 2 * 3?")]

res = llm.invoke(messages)

In [46]:
res.tool_calls

[{'name': 'multiply',
  'args': {'a': 2, 'b': 3},
  'id': '8s8yp5358',
  'type': 'tool_call'}]

In [37]:
{
    "wikipedia": wiki_tool,
    "division": division,
    "multiply": multiply
}["multiply".lower()]

StructuredTool(name='multiply', description='Multiply 2 numbers', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x000001E39FBC5850>)

In [47]:
for tool_call in res.tool_calls:
    selected_tool = {
        "wikipedia": wiki_tool,
        "division": division,
        "multiply": multiply
    }[tool_call["name"].lower()]

    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)

In [48]:
messages

[HumanMessage(content='What is 2 * 3?', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='6', name='multiply', tool_call_id='8s8yp5358')]

In [49]:
llm.invoke(messages).content

'The result of multiplying 2 by 3 is **6**.'